# Day 3：从零手写 DPO 训练（面试 Tier 1！）

> **目标**：从零实现完整的 DPO 训练循环——`偏好数据集构建 → GPT-2 Policy/Ref 初始化 → log_probs 计算 → DPO Loss 实现 → 训练循环 → 隐式 Reward 提取与可视化 → 训练前后生成质量对比`。在模拟偏好数据上训练成功，观察隐式 reward 的分离效果，对比 DPO 训练前后的生成质量变化。
>
> DPO 训练循环是面试 Tier 1 考点——相比 RLHF-PPO 的数百行代码，DPO 只需十几行核心逻辑，但每一行都对应 Day 2 推导的数学公式。

---

## 实现路线图

```
Part 1: 环境准备与数学回顾
  导入库 → 设备设置 → DPO Loss 公式回顾

Part 2: 偏好数据集构建
  构造偏好对 (prompt, chosen, rejected) → Dataset → DataLoader

Part 3: 模型初始化
  GPT-2 → π_θ (policy, 可训练) + π_ref (reference, 冻结)

Part 4: Log Probabilities 计算
  get_log_probs() → 序列 log probability → response mask 处理

Part 5: DPO Loss 实现
  4 组 log_probs → log ratio → margin → sigmoid → loss

Part 6: 完整训练循环
  forward → DPO Loss → backward → 监控 loss / reward margin / accuracy

Part 7: 隐式 Reward 提取与可视化
  β log(π_θ/π_ref) → chosen vs rejected 分布 → 分离度分析

Part 8: 训练前后生成质量对比
  生成 response → 对比 DPO 训练前后的变化

Part 9: 与 RLHF-PPO 的对比分析
  代码量 / 显存 / 训练时间 / 效果 对比
```

In [ ]:
import random
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

try:
    from transformers import GPT2LMHeadModel, GPT2Tokenizer
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers'])
    from transformers import GPT2LMHeadModel, GPT2Tokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## Part 1：DPO Loss 数学回顾

Day 2 推导的 DPO Loss：

$$L_{\text{DPO}}(\theta) = -\mathbb{E}\left[\log \sigma\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right]$$

核心步骤：
1. 计算 $\pi_\theta$ 对 $y_w$ 和 $y_l$ 的 log probability
2. 计算 $\pi_{\text{ref}}$ 对 $y_w$ 和 $y_l$ 的 log probability（无需梯度）
3. 计算 log ratio: $\log \frac{\pi_\theta(y)}{\pi_{\text{ref}}(y)}$
4. 计算 margin: $\beta \cdot (\text{log\_ratio}_w - \text{log\_ratio}_l)$
5. 计算 loss: $-\log \sigma(\text{margin})$

## Part 2：偏好数据集构建

我们构造一组模拟偏好数据，每条包含 `(prompt, chosen, rejected)`。

偏好规则（人为设定，模拟人类偏好）：
- **chosen**（好回答）：详细、有帮助、正面的回答
- **rejected**（坏回答）：简短、无帮助、消极的回答

In [ ]:
PREFERENCE_DATA = [
    {
        "prompt": "What is machine learning?",
        "chosen": "Machine learning is a branch of artificial intelligence that enables computers to learn patterns from data and make predictions without being explicitly programmed.",
        "rejected": "I don't know."
    },
    {
        "prompt": "Explain neural networks.",
        "chosen": "Neural networks are computing systems inspired by the brain, consisting of layers of interconnected nodes that process information and learn to recognize patterns.",
        "rejected": "They are complicated."
    },
    {
        "prompt": "What is deep learning?",
        "chosen": "Deep learning is a subset of machine learning that uses neural networks with many layers to learn hierarchical representations of data, enabling powerful pattern recognition.",
        "rejected": "It's just AI stuff."
    },
    {
        "prompt": "How does gradient descent work?",
        "chosen": "Gradient descent is an optimization algorithm that iteratively adjusts model parameters by moving in the direction of steepest decrease of the loss function.",
        "rejected": "You just minimize the loss."
    },
    {
        "prompt": "What is natural language processing?",
        "chosen": "Natural language processing is a field of AI that focuses on enabling computers to understand, interpret, and generate human language in useful and meaningful ways.",
        "rejected": "It's about text."
    },
    {
        "prompt": "Explain reinforcement learning.",
        "chosen": "Reinforcement learning is a paradigm where an agent learns to make decisions by taking actions in an environment and receiving rewards or penalties as feedback.",
        "rejected": "An agent does stuff and gets rewards."
    },
    {
        "prompt": "What is a transformer model?",
        "chosen": "A transformer is a neural network architecture that uses self-attention mechanisms to process sequential data in parallel, revolutionizing natural language processing.",
        "rejected": "It's a type of model."
    },
    {
        "prompt": "What is overfitting?",
        "chosen": "Overfitting occurs when a model learns the training data too well, including its noise and outliers, causing it to perform poorly on new unseen data.",
        "rejected": "When the model is too good."
    },
    {
        "prompt": "Explain backpropagation.",
        "chosen": "Backpropagation is an algorithm that computes gradients of the loss function with respect to each weight by applying the chain rule layer by layer from output to input.",
        "rejected": "It sends errors backwards."
    },
    {
        "prompt": "What is transfer learning?",
        "chosen": "Transfer learning is a technique where a model trained on one task is repurposed as the starting point for a model on a different but related task, saving time and data.",
        "rejected": "Using a pretrained model."
    },
    {
        "prompt": "How do attention mechanisms work?",
        "chosen": "Attention mechanisms allow models to focus on relevant parts of the input by computing weighted sums, where weights reflect the importance of each element to the current task.",
        "rejected": "They pay attention to things."
    },
    {
        "prompt": "What is a loss function?",
        "chosen": "A loss function measures the difference between a model's predictions and the actual target values, providing a quantitative signal to guide the optimization process.",
        "rejected": "It measures error."
    },
]

print(f"Number of preference pairs: {len(PREFERENCE_DATA)}")
print(f"\nExample:")
print(f"  Prompt:   {PREFERENCE_DATA[0]['prompt']}")
print(f"  Chosen:   {PREFERENCE_DATA[0]['chosen'][:80]}...")
print(f"  Rejected: {PREFERENCE_DATA[0]['rejected']}")

In [ ]:
class PreferenceDataset(Dataset):
    """DPO 偏好数据集：每条数据包含 (prompt, chosen, rejected) 的 token ids"""
    
    def __init__(self, data, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = []
        
        for item in data:
            prompt = item["prompt"]
            chosen = item["chosen"]
            rejected = item["rejected"]
            
            chosen_sample = self._encode(prompt, chosen)
            rejected_sample = self._encode(prompt, rejected)
            
            if chosen_sample is not None and rejected_sample is not None:
                self.samples.append({
                    "chosen_input_ids": chosen_sample["input_ids"],
                    "chosen_attention_mask": chosen_sample["attention_mask"],
                    "chosen_labels_mask": chosen_sample["labels_mask"],
                    "rejected_input_ids": rejected_sample["input_ids"],
                    "rejected_attention_mask": rejected_sample["attention_mask"],
                    "rejected_labels_mask": rejected_sample["labels_mask"],
                })
    
    def _encode(self, prompt, response):
        """将 prompt + response 编码为 token ids，并生成 response 部分的 mask"""
        prompt_text = f"Question: {prompt}\nAnswer: "
        full_text = prompt_text + response
        
        prompt_ids = self.tokenizer.encode(prompt_text, add_special_tokens=False)
        full_ids = self.tokenizer.encode(full_text, add_special_tokens=False)
        
        if len(full_ids) > self.max_length:
            full_ids = full_ids[:self.max_length]
        
        prompt_len = len(prompt_ids)
        seq_len = len(full_ids)
        
        pad_len = self.max_length - seq_len
        input_ids = full_ids + [self.tokenizer.pad_token_id] * pad_len
        attention_mask = [1] * seq_len + [0] * pad_len
        labels_mask = [0] * prompt_len + [1] * (seq_len - prompt_len) + [0] * pad_len
        
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels_mask": torch.tensor(labels_mask, dtype=torch.float),
        }
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

dataset = PreferenceDataset(PREFERENCE_DATA, tokenizer, max_length=128)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

print(f"Dataset size: {len(dataset)}")
sample = dataset[0]
print(f"Chosen input_ids shape:   {sample['chosen_input_ids'].shape}")
print(f"Rejected input_ids shape: {sample['rejected_input_ids'].shape}")
print(f"Chosen labels_mask sum:   {sample['chosen_labels_mask'].sum().item()} tokens")
print(f"Rejected labels_mask sum: {sample['rejected_labels_mask'].sum().item()} tokens")

## Part 3：模型初始化

DPO 需要两个模型：
- **Policy Model** $\pi_\theta$：初始化为 SFT 模型（这里用 GPT-2），训练中更新
- **Reference Model** $\pi_{\text{ref}}$：$\pi_\theta$ 的 deepcopy，训练中冻结

对比 RLHF 的 4 个模型（Actor + Critic + Reference + RM），DPO 只需要 2 个。

In [ ]:
policy_model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)
ref_model = copy.deepcopy(policy_model).to(device)

ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in policy_model.parameters())
trainable_params = sum(p.numel() for p in policy_model.parameters() if p.requires_grad)
ref_params = sum(p.numel() for p in ref_model.parameters())

print(f"Policy Model parameters:  {total_params / 1e6:.1f}M (trainable: {trainable_params / 1e6:.1f}M)")
print(f"Reference Model parameters: {ref_params / 1e6:.1f}M (frozen)")
print(f"Total parameters: {(total_params + ref_params) / 1e6:.1f}M")
print(f"\nCompare with RLHF: would need ~{total_params * 4 / 1e6:.1f}M (4 models)")
print(f"DPO saves ~{(total_params * 4 - total_params * 2) / 1e6:.1f}M parameters ({50}% reduction)")

## Part 4：Log Probabilities 计算

DPO 的核心计算是 **序列 log probability**：

$$\log \pi_\theta(y|x) = \sum_{t=1}^{T} \log P_\theta(y_t \mid x, y_{<t})$$

实现要点：
1. 前向传播得到 logits
2. 对 logits 做 log_softmax 得到 per-token log probs
3. gather 取出实际 token 对应的 log prob
4. 只对 **response 部分**求和（用 labels_mask 过滤 prompt 部分）

In [ ]:
def get_log_probs(model, input_ids, attention_mask, labels_mask):
    """
    计算 response 部分的 log probability 之和。
    
    对应 Day 2 公式:
    log π(y|x) = Σ_t log P(y_t | x, y_{<t})
    
    Args:
        model: 语言模型 (GPT-2)
        input_ids: [batch, seq_len] 完整序列 (prompt + response + padding)
        attention_mask: [batch, seq_len] 有效 token mask
        labels_mask: [batch, seq_len] response 部分的 mask
    
    Returns:
        log_probs: [batch] 每条序列 response 部分的 log prob 之和
    """
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits  # [batch, seq_len, vocab_size]
    
    # shift: position t 的 logits 预测 position t+1 的 token
    # logits[:, :-1, :] 用于预测 input_ids[:, 1:]
    log_probs_all = F.log_softmax(logits[:, :-1, :], dim=-1)  # [batch, seq_len-1, vocab]
    
    # gather: 取出实际 token 对应的 log prob
    target_ids = input_ids[:, 1:]  # [batch, seq_len-1]
    per_token_log_probs = log_probs_all.gather(
        dim=-1, index=target_ids.unsqueeze(-1)
    ).squeeze(-1)  # [batch, seq_len-1]
    
    # 只对 response 部分求和
    response_mask = labels_mask[:, 1:]  # shift mask to align with per_token_log_probs
    masked_log_probs = per_token_log_probs * response_mask
    seq_log_probs = masked_log_probs.sum(dim=-1)  # [batch]
    
    return seq_log_probs

In [ ]:
# 验证 get_log_probs
sample = dataset[0]
test_ids = sample["chosen_input_ids"].unsqueeze(0).to(device)
test_mask = sample["chosen_attention_mask"].unsqueeze(0).to(device)
test_labels = sample["chosen_labels_mask"].unsqueeze(0).to(device)

with torch.no_grad():
    logp = get_log_probs(policy_model, test_ids, test_mask, test_labels)
    logp_ref = get_log_probs(ref_model, test_ids, test_mask, test_labels)

print(f"Policy log P(chosen):    {logp.item():.4f}")
print(f"Reference log P(chosen): {logp_ref.item():.4f}")
print(f"Log ratio (should be ~0 before training): {(logp - logp_ref).item():.6f}")

## Part 5：DPO Loss 实现

对应 Day 2 推导的公式：

$$L_{\text{DPO}} = -\mathbb{E}\left[\log \sigma\left(\beta \left(\log \frac{\pi_\theta(y_w)}{\pi_{\text{ref}}(y_w)} - \log \frac{\pi_\theta(y_l)}{\pi_{\text{ref}}(y_l)}\right)\right)\right]$$

代码中每一行都对应推导中的一步。

In [ ]:
def dpo_loss(policy_model, ref_model, batch, beta=0.1):
    """
    计算 DPO Loss。
    
    对应 Day 2 推导:
    L_DPO = -E[log σ(β(log(π_θ(y_w)/π_ref(y_w)) - log(π_θ(y_l)/π_ref(y_l))))]
    
    Returns:
        loss: scalar, DPO loss
        metrics: dict with monitoring info
    """
    # ====== Step 1: 计算 4 组 log probabilities ======
    # π_θ(y_w|x) — 需要梯度
    policy_logps_chosen = get_log_probs(
        policy_model,
        batch["chosen_input_ids"],
        batch["chosen_attention_mask"],
        batch["chosen_labels_mask"]
    )
    # π_θ(y_l|x) — 需要梯度
    policy_logps_rejected = get_log_probs(
        policy_model,
        batch["rejected_input_ids"],
        batch["rejected_attention_mask"],
        batch["rejected_labels_mask"]
    )
    
    with torch.no_grad():
        # π_ref(y_w|x) — 不需要梯度
        ref_logps_chosen = get_log_probs(
            ref_model,
            batch["chosen_input_ids"],
            batch["chosen_attention_mask"],
            batch["chosen_labels_mask"]
        )
        # π_ref(y_l|x) — 不需要梯度
        ref_logps_rejected = get_log_probs(
            ref_model,
            batch["rejected_input_ids"],
            batch["rejected_attention_mask"],
            batch["rejected_labels_mask"]
        )
    
    # ====== Step 2: 计算 log ratio ======
    # log(π_θ/π_ref) for chosen and rejected
    log_ratio_chosen = policy_logps_chosen - ref_logps_chosen
    log_ratio_rejected = policy_logps_rejected - ref_logps_rejected
    
    # ====== Step 3: 计算 DPO Loss ======
    # β * (log_ratio_chosen - log_ratio_rejected)
    logits = beta * (log_ratio_chosen - log_ratio_rejected)
    
    # -log σ(logits)
    loss = -F.logsigmoid(logits).mean()
    
    # ====== Metrics for monitoring ======
    with torch.no_grad():
        chosen_rewards = beta * log_ratio_chosen
        rejected_rewards = beta * log_ratio_rejected
        reward_margin = (chosen_rewards - rejected_rewards).mean().item()
        accuracy = (logits > 0).float().mean().item()
    
    metrics = {
        "loss": loss.item(),
        "reward_margin": reward_margin,
        "accuracy": accuracy,
        "chosen_reward": chosen_rewards.mean().item(),
        "rejected_reward": rejected_rewards.mean().item(),
        "log_ratio_chosen": log_ratio_chosen.mean().item(),
        "log_ratio_rejected": log_ratio_rejected.mean().item(),
    }
    
    return loss, metrics

In [ ]:
# 验证 DPO Loss（训练前）
policy_model.train()
test_batch = next(iter(dataloader))
test_batch = {k: v.to(device) for k, v in test_batch.items()}

loss, metrics = dpo_loss(policy_model, ref_model, test_batch, beta=0.1)

print("=== DPO Loss before training ===")
print(f"Loss:            {metrics['loss']:.4f}")
print(f"Reward margin:   {metrics['reward_margin']:.4f}  (should be ~0 before training)")
print(f"Accuracy:        {metrics['accuracy']:.2%}")
print(f"Chosen reward:   {metrics['chosen_reward']:.4f}")
print(f"Rejected reward: {metrics['rejected_reward']:.4f}")
print(f"\nNote: before training, policy = ref, so log ratios should be ~0")

## Part 6：完整训练循环

DPO 训练循环的简洁性是其核心优势之一。

对比 RLHF-PPO 需要 rollout → RM scoring → GAE → multi-epoch PPO update，DPO 只需要标准的 forward → loss → backward。

In [ ]:
# ====== 超参数 ======
BETA = 0.1
LR = 1e-5
EPOCHS = 30

optimizer = optim.AdamW(policy_model.parameters(), lr=LR, weight_decay=0.01)

history = {
    "loss": [], "reward_margin": [], "accuracy": [],
    "chosen_reward": [], "rejected_reward": [],
}

print(f"=== DPO Training ===")
print(f"Beta: {BETA}, LR: {LR}, Epochs: {EPOCHS}")
print(f"Dataset size: {len(dataset)}, Batch size: {dataloader.batch_size}")
print(f"{'='*60}")

policy_model.train()
for epoch in range(EPOCHS):
    epoch_metrics = {k: [] for k in history}
    
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        
        loss, metrics = dpo_loss(policy_model, ref_model, batch, beta=BETA)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy_model.parameters(), max_norm=1.0)
        optimizer.step()
        
        for k in epoch_metrics:
            epoch_metrics[k].append(metrics[k])
    
    for k in history:
        avg = np.mean(epoch_metrics[k])
        history[k].append(avg)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
              f"Loss: {history['loss'][-1]:.4f} | "
              f"Margin: {history['reward_margin'][-1]:.4f} | "
              f"Acc: {history['accuracy'][-1]:.2%} | "
              f"R_w: {history['chosen_reward'][-1]:.4f} | "
              f"R_l: {history['rejected_reward'][-1]:.4f}")

print(f"{'='*60}")
print(f"Training complete!")
print(f"Final Loss: {history['loss'][-1]:.4f}")
print(f"Final Accuracy: {history['accuracy'][-1]:.2%}")
print(f"Final Reward Margin: {history['reward_margin'][-1]:.4f}")

In [ ]:
# 训练曲线可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["loss"])
axes[0].set_title("DPO Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["reward_margin"])
axes[1].set_title("Reward Margin (chosen - rejected)")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Margin")
axes[1].axhline(y=0, color='r', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

axes[2].plot(history["chosen_reward"], label="Chosen", color="green")
axes[2].plot(history["rejected_reward"], label="Rejected", color="red")
axes[2].set_title("Implicit Rewards")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Reward")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 7：隐式 Reward 提取与可视化

DPO 训练后，从策略中提取隐式 reward（Day 2 第七节）：

$$\hat{r}_\theta(x, y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}$$

如果训练成功，chosen 的隐式 reward 应该显著高于 rejected。

In [ ]:
def compute_implicit_rewards(policy_model, ref_model, dataset, beta=0.1):
    """提取所有样本的隐式 reward"""
    policy_model.eval()
    chosen_rewards = []
    rejected_rewards = []
    
    with torch.no_grad():
        for i in range(len(dataset)):
            sample = dataset[i]
            
            for response_type in ["chosen", "rejected"]:
                ids = sample[f"{response_type}_input_ids"].unsqueeze(0).to(device)
                mask = sample[f"{response_type}_attention_mask"].unsqueeze(0).to(device)
                labels = sample[f"{response_type}_labels_mask"].unsqueeze(0).to(device)
                
                policy_logp = get_log_probs(policy_model, ids, mask, labels)
                ref_logp = get_log_probs(ref_model, ids, mask, labels)
                
                implicit_reward = beta * (policy_logp - ref_logp)
                
                if response_type == "chosen":
                    chosen_rewards.append(implicit_reward.item())
                else:
                    rejected_rewards.append(implicit_reward.item())
    
    return chosen_rewards, rejected_rewards

chosen_rewards, rejected_rewards = compute_implicit_rewards(
    policy_model, ref_model, dataset, beta=BETA
)

print("=== Implicit Reward Statistics ===")
print(f"Chosen:   mean={np.mean(chosen_rewards):.4f}, std={np.std(chosen_rewards):.4f}")
print(f"Rejected: mean={np.mean(rejected_rewards):.4f}, std={np.std(rejected_rewards):.4f}")
print(f"Margin:   {np.mean(chosen_rewards) - np.mean(rejected_rewards):.4f}")
print(f"Accuracy: {np.mean([c > r for c, r in zip(chosen_rewards, rejected_rewards)]):.2%}")

In [ ]:
# 隐式 Reward 分布可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 直方图
axes[0].hist(chosen_rewards, bins=10, alpha=0.6, label="Chosen", color="green")
axes[0].hist(rejected_rewards, bins=10, alpha=0.6, label="Rejected", color="red")
axes[0].set_title("Implicit Reward Distribution")
axes[0].set_xlabel("Implicit Reward")
axes[0].set_ylabel("Count")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 逐样本对比
x_pos = range(len(chosen_rewards))
axes[1].bar([x - 0.2 for x in x_pos], chosen_rewards, width=0.4, label="Chosen", color="green", alpha=0.7)
axes[1].bar([x + 0.2 for x in x_pos], rejected_rewards, width=0.4, label="Rejected", color="red", alpha=0.7)
axes[1].set_title("Per-Sample Implicit Rewards")
axes[1].set_xlabel("Sample Index")
axes[1].set_ylabel("Implicit Reward")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 8：训练前后生成质量对比

验证 DPO 训练是否改变了模型的生成行为。

In [ ]:
def generate_response(model, tokenizer, prompt, max_new_tokens=50):
    """生成 response"""
    model.eval()
    input_text = f"Question: {prompt}\nAnswer: "
    input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

test_prompts = [
    "What is machine learning?",
    "Explain neural networks.",
    "What is deep learning?",
    "How does gradient descent work?",
]

print("=== Generation Comparison: Reference vs DPO-trained ===")
print("=" * 70)

for prompt in test_prompts:
    ref_response = generate_response(ref_model, tokenizer, prompt)
    dpo_response = generate_response(policy_model, tokenizer, prompt)
    
    print(f"\nPrompt: {prompt}")
    print(f"  [Reference] {ref_response[:120]}")
    print(f"  [DPO]       {dpo_response[:120]}")
    print("-" * 70)

## Part 9：与 RLHF-PPO 的对比分析

### 代码量对比

| 组件 | RLHF-PPO (W10 Day6) | DPO (本 notebook) |
|------|---------------------|-------------------|
| 数据集 | 偏好数据 + prompt 集 | 偏好数据 |
| 模型初始化 | 4 个模型 | 2 个模型 |
| 核心 loss 函数 | ~50 行 (PPO + Value + KL) | ~10 行 |
| 训练循环 | ~100 行 (rollout + GAE + PPO) | ~15 行 |
| 总核心代码 | ~200+ 行 | ~40 行 |

### 训练特性对比

| 维度 | RLHF-PPO | DPO |
|------|---------|-----|
| 训练稳定性 | 不稳定 | 稳定（类似 SFT）|
| 需要 rollout | 是（最慢的部分）| 否 |
| 需要 GAE | 是 | 否 |
| Reward Hacking | 可能 | 不存在 |
| 显存 (7B) | ~140 GB | ~70 GB |

In [ ]:
# 总结
print("=" * 60)
print("Day 3 Summary: DPO Training from Scratch")
print("=" * 60)
print(f"\n1. Dataset: {len(dataset)} preference pairs")
print(f"2. Model: GPT-2 ({total_params/1e6:.1f}M params)")
print(f"3. Training: {EPOCHS} epochs, β={BETA}, lr={LR}")
print(f"4. Final Loss: {history['loss'][-1]:.4f}")
print(f"5. Final Accuracy: {history['accuracy'][-1]:.2%}")
print(f"6. Reward Margin: {history['reward_margin'][-1]:.4f}")
print(f"\nKey takeaways:")
print(f"  - DPO training is as simple as SFT")
print(f"  - Only 2 models needed (vs 4 for RLHF)")
print(f"  - Stable training curve (no KL explosion)")
print(f"  - Implicit rewards separate chosen vs rejected")